# IS 362 Project 2: Tidying Wide Data

**Author:** Shehou Fisiru

This notebook tidies and analyzes three wide datasets from the Week 5 discussion.

In [34]:
import pandas as pd

---
## 1. Retail Sales

I used the fictional data set created by Man Wayne Hui's discussion post. The data tells us monthly units sold by store location, sales channel, and product category.

In [35]:
retail_wide = pd.read_csv("retail_sales_wide.csv")
retail_wide

,Location,Sales Channel,Electronics,Apparel,Home & Kitchen,Sporting Goods
0,Downtown,In-Store,320,580,210,180
1,Downtown,Online Pickup,850,620,340,290
2,Midtown,In-Store,280,490,190,150
3,Midtown,Online Pickup,710,530,280,240
4,Suburb,In-Store,190,340,260,210
5,Suburb,Online Pickup,420,380,310,260


The data is untidy because the product categories are values stored as column headers. They should be one `Category` column.

In order to tidy the data, I `melt()` the four category columns into `Category` and `Units_Sold`.

In [36]:
retail = retail_wide.melt(
    id_vars=["Location", "Sales Channel"],
    var_name="Category",
    value_name="Units_Sold",
)
retail

,Location,Sales Channel,Category,Units_Sold
0,Downtown,In-Store,Electronics,320
1,Downtown,Online Pickup,Electronics,850
2,Midtown,In-Store,Electronics,280
3,Midtown,Online Pickup,Electronics,710
4,Suburb,In-Store,Electronics,190
5,Suburb,Online Pickup,Electronics,420
6,Downtown,In-Store,Apparel,580
7,Downtown,Online Pickup,Apparel,620
8,Midtown,In-Store,Apparel,490
9,Midtown,Online Pickup,Apparel,530


**Analysis:** Which categories sell best in-store vs. online pickup, and which location sells the most in each category?

In [37]:
retail.pivot_table(index="Category", columns="Sales Channel", values="Units_Sold", aggfunc="sum")

Sales Channel,In-Store,Online Pickup
Category,,
Apparel,1410,1530
Electronics,790,1980
Home & Kitchen,660,930
Sporting Goods,540,790


In [38]:
retail.pivot_table(index="Category", columns="Location", values="Units_Sold", aggfunc="sum")

Location,Downtown,Midtown,Suburb
Category,,,
Apparel,1200,1020,720
Electronics,1170,990,610
Home & Kitchen,550,470,570
Sporting Goods,470,390,470


Online pickup outsells in-store in every category, especially Electronics. Apparel is the top in-store category. Downtown leads in Electronics and Apparel, while Suburb leads in Home & Kitchen.

---
## 2. NFL Quarterback Stats

For the second dataset, I used the data provided by Dontae Dawkins' discussion post. He provided passing yards and touchdowns for three quarterbacks over three seasons.

In [39]:
qb_wide = pd.read_csv("nfl_qb_wide.csv", thousands=",") 
qb_wide

,Quarterback,Pass Yards 2023,Pass Yards 2024,Pass Yards 2025,TDs 2023,TDs 2024,TDs 2025
0,Patrick Mahomes,4183,3928,4587,27,26,30
1,Josh Allen,4306,3731,3668,29,28,25
2,Lamar Jackson,3678,4172,4050,24,41,34


The data is untidy because each header like "Pass Yards 2023" combines two variables, the stat type and the season. The data also contains commas, so I used `thousands=","` to read them in as numbers instead of text.

These are the steps I took to tidy the data:
1. `melt()` all six stat columns into one column
2. Split the header into `Stat` and `Season` (split on the last space, since "Pass Yards" has a space in it)
3. `pivot()` so Pass Yards and TDs each get their own column

In [40]:
qb = qb_wide.melt(id_vars="Quarterback", var_name="Stat_Season", value_name="Value")
qb[["Stat", "Season"]] = qb["Stat_Season"].str.rsplit(" ", n=1, expand=True)

qb = qb.pivot(index=["Quarterback", "Season"], columns="Stat", values="Value").reset_index()
qb.columns.name = None
qb

,Quarterback,Season,Pass Yards,TDs
0,Josh Allen,2023,4306,29
1,Josh Allen,2024,3731,28
2,Josh Allen,2025,3668,25
3,Lamar Jackson,2023,3678,24
4,Lamar Jackson,2024,4172,41
5,Lamar Jackson,2025,4050,34
6,Patrick Mahomes,2023,4183,27
7,Patrick Mahomes,2024,3928,26
8,Patrick Mahomes,2025,4587,30


**Analysis:** How did each quarterback's yards and touchdowns change from season to season?

In [41]:
qb["Yards_Change"] = qb.groupby("Quarterback")["Pass Yards"].diff()
qb["TD_Change"] = qb.groupby("Quarterback")["TDs"].diff()
qb

,Quarterback,Season,Pass Yards,TDs,Yards_Change,TD_Change
0,Josh Allen,2023,4306,29,NaN,NaN
1,Josh Allen,2024,3731,28,-575.0,-1.0
2,Josh Allen,2025,3668,25,-63.0,-3.0
3,Lamar Jackson,2023,3678,24,NaN,NaN
4,Lamar Jackson,2024,4172,41,494.0,17.0
5,Lamar Jackson,2025,4050,34,-122.0,-7.0
6,Patrick Mahomes,2023,4183,27,NaN,NaN
7,Patrick Mahomes,2024,3928,26,-255.0,-1.0
8,Patrick Mahomes,2025,4587,30,659.0,4.0


Lamar Jackson improved the most in touchdowns (24 to 34, with 41 in 2024). Mahomes dipped in 2024 but had his best year in 2025. Josh Allen declined in both yards and touchdowns every season.

---
## 3. NBA Team Records

Regular-season records for six NBA teams over the last three seasons.

In [42]:
nba_wide = pd.read_csv("nba_records_wide.csv")
nba_wide

,Team,Conference,2023-24,2024-25,2025-26
0,New York Knicks,East,50-32,51-31,53-29
1,Boston Celtics,East,64-18,61-21,56-26
2,Detroit Pistons,East,14-68,44-38,60-22
3,Oklahoma City Thunder,West,57-25,68-14,64-18
4,San Antonio Spurs,West,22-60,34-48,62-20
5,Denver Nuggets,West,57-25,50-32,54-28


The data is untidy because the seasons are values stored as column headers, and each record like "64-18" packs wins and losses into one cell.

In order to tidy it, I `melt()` the seasons into a `Season` column, then split the record into numeric `Wins` and `Losses`.

In [43]:
nba = nba_wide.melt(id_vars=["Team", "Conference"], var_name="Season", value_name="Record")
nba[["Wins", "Losses"]] = nba["Record"].str.split("-", expand=True).astype(int)
nba = nba.drop(columns="Record")
nba

,Team,Conference,Season,Wins,Losses
0,New York Knicks,East,2023-24,50,32
1,Boston Celtics,East,2023-24,64,18
2,Detroit Pistons,East,2023-24,14,68
3,Oklahoma City Thunder,West,2023-24,57,25
4,San Antonio Spurs,West,2023-24,22,60
5,Denver Nuggets,West,2023-24,57,25
6,New York Knicks,East,2024-25,51,31
7,Boston Celtics,East,2024-25,61,21
8,Detroit Pistons,East,2024-25,44,38
9,Oklahoma City Thunder,West,2024-25,68,14


**Analysis:** How did each team's performance change over the last 3 seasons?

In [44]:
nba["Win_Pct"] = (nba["Wins"] / 82).round(3)
nba.pivot(index="Team", columns="Season", values="Win_Pct")

Season,2023-24,2024-25,2025-26
Team,,,
Boston Celtics,0.780,0.744,0.683
Denver Nuggets,0.695,0.610,0.659
Detroit Pistons,0.171,0.537,0.732
New York Knicks,0.610,0.622,0.646
Oklahoma City Thunder,0.695,0.829,0.780
San Antonio Spurs,0.268,0.415,0.756


The Pistons (14 to 60 wins) and Spurs (22 to 62 wins) had the biggest turnarounds. The Celtics declined each season, and the Knicks improved slightly each year. 